# 6장 실습 — RAPTOR 직접 짜기 (채점)

직접 짜는 셋 중 두 번째입니다. 채울 파일은 **`labs/ch06_raptor.py`** 입니다.

3장의 최단경로와 다른 점이 하나 있습니다.
버스는 아무 때나 출발하지 않습니다. 시간표에 적힌 시각에만 출발합니다.
그래서 "가장 가까운 이웃"이 아니라 "지금 시각에 탈 수 있는 가장 이른 차"를 찾아야 합니다.

순서는 3장과 같습니다. 손으로 답을 아는 시간표에서 시작합니다.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import banner, expect
from smartmob.viz import use_korean_font

use_korean_font()

import ch06_raptor as sol      # 여러분이 채우는 파일

INF = float("inf")

## 1. 손으로 답을 아는 시간표

정류장 다섯 개, 노선 두 개, 운행 세 개입니다.

```
  A ──(1호선)──► B ──(1호선)──► C      08:00 A → 08:10 B → 08:20 C
                 │                     08:30 A → 08:40 B → 08:50 C
             도보 약 100m
                 │
                 D ──(2호선)──► E      08:15 D → 08:25 E
```

A에서 8시에 출발하면 이렇습니다.

| 목적지 | 도착 | 이유 |
|---|---|---|
| C | 08:20 | 1호선 직통 |
| E | 08:25 | B에서 내려 D로 걸어가 2호선 |

8시 5분에 출발하면 첫 차를 놓쳐 C가 08:50이 됩니다.
종이에 그려 놓고 확인할 수 있는 크기입니다. 여기서 막히면 어디가 틀렸는지 바로 보입니다.

In [ ]:
from smartmob.testing import toy_feed

feed = toy_feed()
for name, table in feed.items():
    print(f"{name:12s} {len(table):>4}행")

feed["stop_times"]

## 2. 자료구조 만들기 (교재 6.2 ~ 6.4)

`TransitData.from_gtfs` 를 채우고 아래를 실행합니다.
노선이 아니라 **패턴** 으로 묶는 것이 핵심입니다.
같은 노선이라도 정류장 순서가 다르면 다른 패턴입니다.

In [ ]:
banner("작은 시간표로 자료구조 만들기")
try:
    toy = sol.TransitData.from_gtfs(feed)
    print(f"[v] 정류장 {len(toy.stop_ids)}개, 패턴 {len(toy.patterns)}개")
    for i, p in enumerate(toy.patterns):
        print(f"    패턴 {i}: {p.name}  정류장 {[toy.stop_ids[s] for s in p.stops]}"
              f"  운행 {len(p.departures)}회")
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)

## 3. 탐색 (교재 6.5)

`raptor` 를 채우고 아래를 실행합니다.
돌려주는 것은 `best` 리스트 하나입니다. `best[i]` 가 정류장 i 에 가장 이른 도착시각(초)입니다.

In [ ]:
def hhmm(seconds):
    if seconds == INF:
        return "못 감"
    return f"{int(seconds) // 3600:02d}:{int(seconds) % 3600 // 60:02d}"


def at(data, stop_id):
    """정류장 id 를 인덱스로. `index_of` 를 안 만들었어도 동작합니다."""
    index = getattr(data, "index_of", None)
    if index and stop_id in index:
        return index[stop_id]
    return list(data.stop_ids).index(stop_id)


banner("작은 시간표 탐색")
try:
    toy = sol.TransitData.from_gtfs(feed)
    a, c, e = at(toy, "A"), at(toy, "C"), at(toy, "E")

    best = sol.raptor(toy, [(a, 0)], 8 * 3600)
    expect("A 08:00 출발 → C 도착", hhmm(best[c]), "08:20")
    expect("A 08:00 출발 → E 도착", hhmm(best[e]), "08:25")

    late = sol.raptor(toy, [(a, 0)], 8 * 3600 + 5 * 60)
    expect("A 08:05 출발 → C 도착", hhmm(late[c]), "08:50")

    night = sol.raptor(toy, [(a, 0)], 23 * 3600)
    expect("A 23:00 출발 → C 도착", hhmm(night[c]), "못 감")
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

네 줄이 전부 통과하면 알고리즘이 맞는 것입니다.
남은 것은 실제 데이터에서도 견디는지입니다.

## 4. 실제 하남 GTFS (교재 6.7)

정류장이 5개에서 4,203개로 늘어납니다. 알고리즘은 그대로입니다.

In [ ]:
from smartmob.data import load_gtfs

hanam = load_gtfs("hanam")
print({k: len(v) for k, v in hanam.items()})

In [ ]:
import time

banner("하남 GTFS")
try:
    t0 = time.perf_counter()
    data = sol.TransitData.from_gtfs(hanam)
    print(f"[v] 자료구조 {time.perf_counter() - t0:.1f}초, "
          f"정류장 {len(data.stop_ids):,}개, 패턴 {len(data.patterns):,}개")

    origins = data.access_stops(37.5393, 127.2148)     # 하남시청
    print(f"    출발 후보 정류장 {len(origins)}곳")

    t0 = time.perf_counter()
    best = sol.raptor(data, origins, 8 * 3600)
    reached = sum(1 for t in best if t < INF)
    print(f"[v] 탐색 {time.perf_counter() - t0:.2f}초, "
          f"{reached:,}개 도달 ({reached / len(best):.0%})")
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

## 5. 불변식 확인 (교재 6.7)

실제 데이터에는 손으로 아는 정답이 없습니다.
대신 **반드시 성립해야 하는 성질** 을 확인합니다.
30분 늦게 출발했는데 더 일찍 도착하는 정류장이 있으면 어딘가 틀린 것입니다.

In [ ]:
try:
    data = sol.TransitData.from_gtfs(hanam)
    origins = data.access_stops(37.5393, 127.2148)
    early = sol.raptor(data, origins, 8 * 3600)
    later = sol.raptor(data, origins, 8 * 3600 + 1800)

    bad = sum(1 for a, b in zip(early, later) if a < INF and b < INF and b < a)
    expect("늦게 출발했는데 더 일찍 도착한 정류장", bad, 0)
except NotImplementedError as exc:
    print("[ ] 아직 구현하지 않았습니다 —", exc)
except Exception as exc:
    print(f"[x] {type(exc).__name__}: {exc}")

## 6. 채점

In [ ]:
from check import check

report = check("ch06")

## 7. 경로 복원해 보기 (교재 6.6)

`best` 는 도착시각만 알려 줍니다. 어느 버스를 어디서 탔는지는 따로 되짚어야 합니다.
복원은 채점 대상이 아니라서, 여기서는 교재의 정돈본(`smartmob.teaching.raptor`)으로 봅니다.

In [ ]:
from smartmob.teaching.raptor import TransitData, journey, raptor, summarize

ref_data = TransitData.from_gtfs(hanam)
ref_origins = ref_data.access_stops(37.5393, 127.2148)     # 하남시청
result = raptor(ref_data, ref_origins, 8 * 3600)

near_misa = ref_data.access_stops(37.5606, 127.1930)
target = near_misa[0][0]                                   # 미사역 근처 정류장
legs = journey(ref_data, result, target)
print(f"미사역 근처 정류장 {len(near_misa)}곳 중 {ref_data.stop_names[target]} 로 갑니다")

for leg in legs:
    if leg["kind"] == "transit":
        print(f"  {leg['mode']:7s} {leg['route']:12s} "
              f"{hhmm(leg['board_time'])} → {hhmm(leg['alight_time'])}  "
              f"{leg['n_stops']}정거장 {leg['km']}km")
    else:
        print(f"  WALK    {'':12s} {leg['seconds'] / 60:.1f}분  {leg['km']}km")

summarize(ref_data, legs, 8 * 3600)

## 제출할 것

1. 채운 `labs/ch06_raptor.py`
2. 채점 셀의 출력 (전부 PASS)
3. 막혔던 지점과 어떻게 풀었는지 3~5줄

## 정리

- 노선이 아니라 패턴으로 묶습니다. 정류장 순서가 다르면 다른 패턴입니다
- 라운드 k 는 "환승 k-1 번"에 대응합니다. 타는 판단에는 직전 라운드 값을 씁니다
- 큰 데이터에는 정답이 없습니다. 대신 불변식으로 검증합니다
- 7장 실습에서는 여기에 환승 제한과 요금을 붙입니다